# Imports

In [2]:
!uv

An extremely fast Python package manager.

Usage: uv [OPTIONS] <COMMAND>

Commands:
  auth       Manage authentication
  run        Run a command or script
  init       Create a new project
  add        Add dependencies to the project
  remove     Remove dependencies from the project
  version    Read or update the project's version
  sync       Update the project's environment
  lock       Update the project's lockfile
  export     Export the project's lockfile to an alternate format
  tree       Display the project's dependency tree
  format     Format Python code in the project
  check      Run checks on the project
  audit      Audit the project's dependencies
  tool       Run and install commands provided by Python packages
  python     Manage Python versions and installations
  pip        Manage Python packages with a pip-compatible interface
  venv       Create a virtual environment
  build      Build Python packages into source distributions and wheels
  publish    Upload distr

# Architecture

## Audio Encoder

In [ ]:



import math
from collections.abc import Sequence
import torch
from beartype import beartype
from einops import rearrange
from jaxtyping import Float, jaxtyped
from torch import Tensor, nn
from torch.nn import functional
from architecture.pretraining.kda import KimiDeltaAttention


class CausalLocalAttention(nn.Module):
    """
    causal swiAttn. still (qk^{T}/sqrt(d_h))V but windowed out using a logical and from torch.
    takes ideas from https://arxiv.org/html/2505.02074v1
    @param hidden_dimension int for the latent space to occupy
    @param heads: attn heads
    @param window_steps


    """
    def __init__(self, hidden_dimension: int, heads: int, window_steps: int) -> None:
        super().__init__()
        if hidden_dimension % heads or window_steps <= 0:
            raise ValueError(
                "Attention requires divisible dimensions and a positive window"
            )
        self.heads = heads
        self.window_steps = window_steps
        self.query_projection = nn.Linear(
            hidden_dimension, hidden_dimension, bias=False
        )
        self.key_projection = nn.Linear(hidden_dimension, hidden_dimension, bias=False)
        self.value_projection = nn.Linear(
            hidden_dimension, hidden_dimension, bias=False
        )
        self.output_projection = nn.Linear(
            hidden_dimension, hidden_dimension, bias=False
        )

    @jaxtyped(typechecker=beartype)
    def forward(
        self, tokens: Float[Tensor, "batch time hidden"]
    ) -> Float[Tensor, "batch time hidden"]:
        queries = rearrange(
            self.query_projection(tokens),
            "batch time (head channel) -> batch head time channel",
            head=self.heads,
        )
        keys = rearrange(
            self.key_projection(tokens),
            "batch time (head channel) -> batch head time channel",
            head=self.heads,
        )
        values = rearrange(
            self.value_projection(tokens),
            "batch time (head channel) -> batch head time channel",
            head=self.heads,
        )
        positions = torch.arange(tokens.shape[1], device=tokens.device)
        distance = positions[:, None] - positions[None, :]
        attention_mask = torch.logical_and(
            distance >= 0,
            distance < self.window_steps,
        )
        attended = functional.scaled_dot_product_attention(
            queries,
            keys,
            values,
            attn_mask=attention_mask,
        )
        return self.output_projection(
            rearrange(
                attended,
                "batch head time channel -> batch time (head channel)",
            )
        )


class HybridAudioEncoder(nn.Module):
    def __init__(
        self,
        input_channels: int,
        hidden_dimension: int,
        latent_dimension: int,
        downsample_strides: Sequence[int],
        block_pattern: Sequence[str],
        kda_heads: int,
        kda_head_dimension: int,
        kda_chunk_size: int,
        kda_minimum_log_decay: float,
        attention_heads: int,
        attention_window_steps: int,
    ) -> None:
        """
        setup for waveform -> latent. best way is linear/softmax attn with some ratio
        alternates between kimi delta and transformer layers
        operates on channel_dim
        ratio can be passed as seq but maybe better to parse as a frac non-natively?

        @param input_channels: int how many channels which are for mlp/transformer in
        @param h_dim: int dim of conv output for downsampling waveform
        @param latent_dim: int what dim/size of emb space
        @param downsample_strides: seq of int representing strides of conv for each. need to do a len check
        @param block_pattern: seq(str) like "K","K","K","A","K","K","K","A" for the pattern of blocks of kda vs attn
        @param kda_heads: num_heads for kda
        @param kda_head_dimension:
        @param kda_chunk_size: size of chunk to perform Linear ops on for KDA
        @param kda_minimum_log_decay: smallest amount to log-decay the logits of kda
        @param attention_heads: how many heads to use in softmax attn
        @param attn_window_steps: since we are using causal attn (still softmax attn, but a little faster than naive SDPA) how big do we want the sliding window steps to be?
        """
        super().__init__()
        if not downsample_strides or any(stride <= 0 for stride in downsample_strides):
            raise ValueError("Audio downsample strides must be positive")

        stem_layers: list[nn.Module] = []
        stem_input_channels = input_channels
        for stride in downsample_strides:
            stem_layers.extend(
                (
                    nn.Conv1d(
                        stem_input_channels,
                        hidden_dimension,
                        kernel_size=stride,
                        stride=stride,
                    ),
                    nn.GELU(),
                )
            )
            stem_input_channels = hidden_dimension
        self.stem = nn.Sequential(*stem_layers)
        self.total_stride = math.prod(downsample_strides)

        # whole concept of mixers comes I think from Mamba?
        # https://github.com/state-spaces/mamba/blob/main/mamba_ssm/models/mixer_seq_simple.py
        mixers: list[nn.Module] = []
        for block_kind in block_pattern:
            if block_kind == "kda":
                mixers.append(
                    KimiDeltaAttention(
                        hidden_dimension,
                        kda_heads,
                        kda_head_dimension,
                        kda_chunk_size,
                        kda_minimum_log_decay,
                    )
                )
            elif block_kind == "attention":
                mixers.append(
                    CausalLocalAttention(
                        hidden_dimension,
                        attention_heads,
                        attention_window_steps,
                    )
                )
            else:
                raise ValueError(f"Unknown audio mixer: {block_kind}")
        self.mixers = nn.ModuleList(mixers)
        self.mixer_norms = nn.ModuleList(
            nn.RMSNorm(hidden_dimension) for block_kind in block_pattern
        )
        self.feed_forward_norms = nn.ModuleList(
            nn.RMSNorm(hidden_dimension) for block_kind in block_pattern
        )
        self.feed_forwards = nn.ModuleList(
            nn.Sequential(
                nn.Linear(hidden_dimension, hidden_dimension * 4),
                nn.SiLU(),
                nn.Linear(hidden_dimension * 4, hidden_dimension),
            )
            for block_kind in block_pattern
        )
        self.output_projection = nn.Linear(hidden_dimension, latent_dimension)
        self.kda_block_count = block_pattern.count("kda")

    @jaxtyped(typechecker=beartype)
    def forward(
        self,
        waveform: Float[Tensor, "batch channel sample"],
        initial_states: list[Tensor] | None = None,
    ) -> tuple[Float[Tensor, "batch time latent"], list[Tensor]]:
        """
        takes in waveform (tensor) with some initial states and runs fwd pass.
        @param waveform: float tensor of shape bxcxsample (or audio_sample, depending on which file you look at)
        @param initial_states: list of tensors for initializing the kda state matrix. can be optional.

        Returns:
            latent representation which is 3 tensor of b x t x l
            l is inherited from the model class, same as t
            also returns the state matrix for next states
        """
        if waveform.shape[-1] % self.total_stride:
            raise ValueError(
                f"Audio samples must be divisible by encoder stride {self.total_stride}"
            )
        if initial_states is not None and len(initial_states) != self.kda_block_count:
            raise ValueError(f"Expected {self.kda_block_count} KDA states")

        tokens = rearrange(
            self.stem(waveform), "batch hidden time -> batch time hidden"
        )
        next_states: list[Tensor] = []
        # I am not sure if there is a "smarter" way to do iteration through this than a counter and enumerate. maybe something with zip?
        # for now, counter = 0, counter+=1 if valid idx else throw err
        state_index = 0
        for mixer_index, mixer in enumerate(self.mixers):
            normalized = self.mixer_norms[mixer_index](tokens)
            if isinstance(mixer, KimiDeltaAttention):
                initial_state = (
                    None if initial_states is None else initial_states[state_index]
                )
                mixed, next_state = mixer(normalized, initial_state)
                next_states.append(next_state)
                state_index += 1
            elif isinstance(mixer, CausalLocalAttention):
                mixed = mixer(normalized)
            else:
                raise TypeError(f"Unsupported mixer type: {type(mixer).__name__}")
            tokens = tokens + mixed
            tokens = tokens + self.feed_forwards[mixer_index](
                self.feed_forward_norms[mixer_index](tokens)
            )
        return self.output_projection(tokens), next_states

## Audio Decoder
(for seeing what our model is hearing, not used during optimization)

In [ ]:
from beartype import beartype
from einops import rearrange
from jaxtyping import Float, jaxtyped
from torch import Tensor, nn


class CausalAudioDecoder(nn.Module):
    def __init__(
        self, latent_dimension: int, output_channels: int, samples_per_token: int
    ) -> None:
        super().__init__()
        if output_channels <= 0 or samples_per_token <= 0:
            raise ValueError("Audio decoder dimensions must be positive")
        self.output_channels = output_channels
        self.samples_per_token = samples_per_token
        self.output_projection = nn.Linear(
            latent_dimension, output_channels * samples_per_token
        )

    @jaxtyped(typechecker=beartype)
    def forward(
        self,
        latents: Float[Tensor, "batch time latent"],
    ) -> Float[Tensor, "batch channel sample"]:
        samples = self.output_projection(latents)
        return rearrange(
            samples,
            "batch time (channel sample) -> batch channel (time sample)",
            channel=self.output_channels,
            sample=self.samples_per_token,
        )

## Video Encoder


In [ ]:
from collections.abc import Sequence
from contextlib import nullcontext
from typing import cast

import timm
import torch
from beartype import beartype
from einops import rearrange, reduce
from jaxtyping import Float, jaxtyped
from timm.models.vision_transformer import VisionTransformer
from torch import Tensor, nn
from torch.nn import functional


class DinoVideoEncoder(nn.Module):
    def __init__(
        self,
        model_id: str,
        pretrained: bool,
        image_size: int,
        layer_numbers: Sequence[int],
        mean: Sequence[float],
        standard_deviation: Sequence[float],
        frozen: bool = True,
    ) -> None:
        """
        in general, dinoV2/V3 returns mean pooled lat of cls_tok, r_1 .. r_4
        you can check fn_signature and look at a sample output
        norm is imagenet norm usually
        but could also be 3 ch of 0.5 (not really standard)
        """
        super().__init__()
        if not layer_numbers or min(layer_numbers) < 1:
            raise ValueError("DINO layer numbers are one-based and must be positive")
        if len(mean) != 3 or len(standard_deviation) != 3:
            raise ValueError("DINO normalization requires three channels")
        self.backbone = cast(
            VisionTransformer,
            timm.create_model(
                model_id,
                pretrained=pretrained,
                num_classes=0,
                img_size=image_size,
            ),
        )
        self.layer_indices = [layer_number - 1 for layer_number in layer_numbers]
        self.image_size = image_size
        self.register_buffer(
            "mean",
            torch.tensor(mean, dtype=torch.float32)[None, :, None, None],
            persistent=False,
        )
        self.register_buffer(
            "standard_deviation",
            torch.tensor(standard_deviation, dtype=torch.float32)[None, :, None, None],
            persistent=False,
        )
        self.frozen = frozen
        self.requires_grad_(not frozen)
        if frozen:
            self.eval()

    @jaxtyped(typechecker=beartype)
    def forward(
        self, frames: Float[Tensor, "batch time channel height width"]
    ) -> Float[Tensor, "batch time patch feature"]:
        batch_size, time_steps = frames.shape[:2]
        images = rearrange(
            frames,
            "batch time channel height width -> (batch time) channel height width",
        )
        if images.shape[-2:] != (self.image_size, self.image_size):
            images = functional.interpolate(
                images,
                size=(self.image_size, self.image_size),
                mode="bilinear",
                align_corners=False,
            )
        images = (images - self.mean) / self.standard_deviation
        gradient_context = torch.no_grad() if self.frozen else nullcontext()
        with gradient_context:
            layer_features = cast(
                list[Tensor],
                self.backbone.forward_intermediates(
                    images,
                    indices=self.layer_indices,
                    norm=True,
                    output_fmt="NLC",
                    intermediates_only=True,
                ),
            )
            patch_features = reduce(
                torch.stack(layer_features),
                "layer image patch feature -> image patch feature",
                "mean",
            )
        return rearrange(
            patch_features,
            "(batch time) patch feature -> batch time patch feature",
            batch=batch_size,
            time=time_steps,
        )

## Video Decoder

In [ ]:
lections.abc import Sequence

from beartype import beartype
from einops import rearrange
from jaxtyping import Float, jaxtyped
from torch import Tensor, nn
from torch.nn import functional


class DecoderResidualBlock(nn.Module):
    def __init__(self, channels: int, normalization_groups: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(normalization_groups, channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(normalization_groups, channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, features: Tensor) -> Tensor:
        return features + self.block(features)


class DecoderUpsampleBlock(nn.Module):
    def __init__(
        self, input_channels: int, output_channels: int, normalization_groups: int
    ) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(input_channels, output_channels, 3, padding=1),
            nn.GroupNorm(normalization_groups, output_channels),
            nn.SiLU(),
        )

    def forward(self, features: Tensor) -> Tensor:
        features = functional.interpolate(
            features,
            scale_factor=2.0,
            mode="bilinear",
            align_corners=False,
        )
        return self.block(features)


class DinoVideoDecoder(nn.Module):
    def __init__(
        self,
        feature_dimension: int,
        hidden_dimension: int,
        upsample_channels: Sequence[int],
        residual_blocks: int,
        normalization_groups: int,
        patch_grid_size: int,
    ) -> None:
        super().__init__()
        channels = (hidden_dimension, *upsample_channels)
        if any(channel % normalization_groups for channel in channels):
            raise ValueError(
                "Decoder channels must divide evenly into normalization groups"
            )
        if residual_blocks <= 0 or patch_grid_size <= 0:
            raise ValueError("Decoder block count and patch grid must be positive")
        self.patch_grid_size = patch_grid_size
        self.input_projection = nn.Conv2d(feature_dimension, hidden_dimension, 1)
        self.residual_stack = nn.Sequential(
            *(
                DecoderResidualBlock(hidden_dimension, normalization_groups)
                for block_index in range(residual_blocks)
            )
        )
        decoder_blocks: list[nn.Module] = []
        input_channels = hidden_dimension
        for output_channels in upsample_channels:
            decoder_blocks.extend(
                (
                    DecoderUpsampleBlock(
                        input_channels,
                        output_channels,
                        normalization_groups,
                    ),
                    DecoderResidualBlock(output_channels, normalization_groups),
                )
            )
            input_channels = output_channels
        self.upsampler = nn.Sequential(*decoder_blocks)
        self.output_head = nn.Sequential(
            nn.Conv2d(input_channels, input_channels, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(input_channels, 3, 3, padding=1),
            nn.Sigmoid(),
        )

    @jaxtyped(typechecker=beartype)
    def forward(
        self,
        latents: Float[Tensor, "batch time patch feature"],
    ) -> Float[Tensor, "batch time channel height width"]:
        if latents.shape[2] != self.patch_grid_size**2:
            raise ValueError(f"Expected {self.patch_grid_size**2} DINO patch tokens")
        feature_maps = rearrange(
            latents,
            "batch time (height width) feature -> (batch time) feature height width",
            height=self.patch_grid_size,
            width=self.patch_grid_size,
        )
        images = self.output_head(
            self.upsampler(self.residual_stack(self.input_projection(feature_maps)))
        )
        return rearrange(
            images,
            "(batch time) channel height width -> batch time channel height width",
            batch=latents.shape[0],
            time=latents.shape[1],
        )

## Wide DDT Head

In [ ]:
from torch import Tensor, nn

type StateDictionary = dict[str, Tensor]
type Checkpoint = dict[str, StateDictionary]
type ModelModules = dict[str, nn.Module]

## Kimi Delta Attention

## Types for Encoders/Decoders

## Transformer

## Latent Action Model

## Hybrid Action Model

## AV Flow World Model

# Loss Functions

## Audio Recon Loss
Not used in optimization
MSE between the "next waveform section prediction" (causal waveform from decoder) and ground truth portion of waveform

In [ ]:
@jaxtyped(typechecker=beartype)
def audio_reconstruction_loss(
    decoder: CausalAudioDecoder,
    latents: Float[Tensor, "batch time latent"],
    waveform: Float[Tensor, "batch channel sample"],
) -> Float[Tensor, ""]:
    reconstruction = decoder(latents)
    if reconstruction.shape != waveform.shape:
        raise ValueError(f"Audio reconstruction shape {reconstruction.shape} does not match {waveform.shape}")
    return torch.mean((reconstruction - waveform).square())

## Image Recon Loss

In [ ]:
@jaxtyped(typechecker=beartype)
def image_reconstruction_loss(
    reconstruction: Float[Tensor, "... channel height width"],
    target: Float[Tensor, "... channel height width"],
) -> Float[Tensor, ""]:
    """
    literally recon loss between a predicted image and a ground truth image sample
    mse, but the nn.MSELoss() returns a callable which we don't want

    @param reconstruction: float tensor with n-ary args but HAS to have channel, width, height
    @param target: float tensor with n-ary args but HAS to have channel, width, height

    Returns: MSE between the recon and target
    """
    return torch.mean((reconstruction - target).square())

## Video Recon Loss

In [ ]:
@jaxtyped(typechecker=beartype)
def video_reconstruction_loss(
    decoder: DinoVideoDecoder,
    latents: Float[Tensor, "batch time patch feature"],
    frames: Float[Tensor, "batch time channel height width"],
) -> Float[Tensor, ""]:
    """
    for a given video, we can decode its latents. however, the decoding can be
    bad. this can be optimized with a reconstruction loss on its frames.
    see image.py for the impl (hint, its mse between ground truth and prediction)

    process is we take in latents, pass them into the decoder (which is a pretrained DinoV3 model) which outputs video frames.

    these frames are a tensor of shape batch x time x channel x height x width, where time
    corresponds to the current second.

    We have made the design decision to operate at 30fps, so time is always between 0 and 29

    actual video frames are also from datasets which are 30fps, so we align them and run this recon loss

    @param decoder: simple decoder which outputs a heatmap from dino latents. dino latents take the form of patches x features over timesteps (see next param)
    @param latents: float tensor of shape batch x time x patch x feature
    @param frames: ground truth float tensor of shape batch x time x channels x height x width which is compared to the output of the decoder(latents)

    Returns (mse(decoder(latents), frames))
    """

    reconstruction = decoder(latents)
    if reconstruction.shape != frames.shape:
        raise ValueError("Video reconstruction and target shapes must match")
    return image_reconstruction_loss(reconstruction, frames)

## Flow Matching AudioVisual Loss Function

### Flow Matching Types

In [ ]:
from dataclasses import dataclass
from typing import Literal

from pydantic import (
    BaseModel,
    ConfigDict,
    Field,
    NonNegativeFloat,
    PositiveFloat,
)
from torch import Tensor


class TimeDistributionConfig(BaseModel):
    model_config = ConfigDict(frozen=True)

    type: Literal["uniform", "logit_normal"] = "logit_normal"
    shift: PositiveFloat = 4.0
    mean: float = 0.0
    standard_deviation: PositiveFloat = 1.0


class TransportConfig(BaseModel):
    model_config = ConfigDict(frozen=True)

    noise_standard_deviation: PositiveFloat = 1.0
    audio_delay: float = Field(default=0.25, ge=0.0, lt=1.0)
    sde_diffusion_scale: NonNegativeFloat = 1.0
    time_distribution: TimeDistributionConfig = TimeDistributionConfig()


@dataclass(frozen=True)
class AudioVisualInterpolant:
    video_state: Tensor
    audio_state: Tensor
    video_velocity: Tensor
    audio_velocity: Tensor
    time: Tensor

### ODE Solver and SDE Solver

In [ ]:
# initial code taken from https://github.com/facebookresearch/Flow-World-Models/blob/main/transport/integrators.py


@jaxtyped(typechecker=beartype)
def audio_clock(
    time: Float[Tensor, "batch"],  # noqa: F821
    delay: float,
) -> tuple[Float[Tensor, "batch"], Float[Tensor, "batch"]]:  # noqa: F821
    """
    not sure I liek clock but tired and lazy

    @param time: float tensor which is batch shaped. created as a timestep embedder
    @param delay: how long to "delay" audio for , since we stream the av

    p = t-d/(1-d) in [0.,1.]

    d/dt = ((t>d) bool mask passed over time tensor) / 1.0 - d)

    Returns
        p, d/dt
    """
    progress = ((time - delay) / (1.0 - delay)).clamp(0.0, 1.0)
    derivative = (time > delay).to(time) / (1.0 - delay)
    return progress, derivative


@jaxtyped(typechecker=beartype)
def flow_map_jump(
    video_state: Float[Tensor, "batch time patch video_feature"],
    audio_state: Float[Tensor, "batch time audio_feature"],
    video_velocity: Float[Tensor, "batch time patch video_feature"],
    audio_velocity: Float[Tensor, "batch time audio_feature"],
    start_time: Float[Tensor, "batch"],  # noqa: F821
    end_time: Float[Tensor, "batch"],  # noqa: F821
) -> tuple[
    Float[Tensor, "batch time patch video_feature"],
    Float[Tensor, "batch time audio_feature"],
]:
    if torch.any(end_time < start_time):
        raise ValueError("Flow-map end time must not precede start time")
    time_gap = end_time - start_time
    return (
        video_state + rearrange(time_gap, "batch -> batch 1 1 1") * video_velocity,
        audio_state + rearrange(time_gap, "batch -> batch 1 1") * audio_velocity,
    )


@torch.no_grad()
@jaxtyped(typechecker=beartype)
def solve_flow_map(
    predictor: AudioVisualFlowPredictor,
    video_state: Float[Tensor, "batch time patch video_feature"],
    audio_state: Float[Tensor, "batch time audio_feature"],
    context_video: Float[Tensor, "batch context_time patch video_feature"],
    context_audio: Float[Tensor, "batch context_time audio_feature"],
    actions: Float[Tensor, "batch time action"] | None,
    integration_grid: Float[Tensor, "grid"],  # noqa: F821
    diagonal: bool,
    context_actions: Float[Tensor, "batch context_time action"] | None = None,
) -> tuple[
    Float[Tensor, "batch time patch video_feature"],
    Float[Tensor, "batch time audio_feature"],
]:
    for integration_index in range(integration_grid.shape[0] - 1):
        start_time = integration_grid[integration_index].expand(video_state.shape[0])
        end_time = integration_grid[integration_index + 1].expand(video_state.shape[0])
        model_end_time = start_time if diagonal else end_time
        video_velocity, audio_velocity = predictor(
            video_state,
            audio_state,
            start_time,
            model_end_time,
            context_video,
            context_audio,
            actions,
            context_actions,
        )
        video_state, audio_state = flow_map_jump(
            video_state,
            audio_state,
            video_velocity,
            audio_velocity,
            start_time,
            end_time,
        )
    return video_state, audio_state


@torch.no_grad()
@jaxtyped(typechecker=beartype)
def solve_sde(
    predictor: AudioVisualFlowPredictor,
    video_state: Float[Tensor, "batch time patch video_feature"],
    audio_state: Float[Tensor, "batch time audio_feature"],
    context_video: Float[Tensor, "batch context_time patch video_feature"],
    context_audio: Float[Tensor, "batch context_time audio_feature"],
    actions: Float[Tensor, "batch time action"] | None,
    integration_grid: Float[Tensor, "grid"],  # noqa: F821
    audio_delay: float,
    noise_standard_deviation: float,
    diffusion_scale: float,
) -> tuple[
    Float[Tensor, "batch time patch video_feature"],
    Float[Tensor, "batch time audio_feature"],
]:
    noise_variance = noise_standard_deviation**2
    epsilon = torch.finfo(video_state.dtype).eps
    for integration_index in range(integration_grid.shape[0] - 1):
        start_time = integration_grid[integration_index].expand(video_state.shape[0])
        end_time = integration_grid[integration_index + 1].expand(video_state.shape[0])
        video_velocity, audio_velocity = predictor(
            video_state,
            audio_state,
            start_time,
            start_time,
            context_video,
            context_audio,
            actions,
        )

        video_progress = rearrange(start_time, "batch -> batch 1 1 1")
        video_remaining = 1.0 - video_progress
        video_noise = video_state - video_progress * video_velocity
        video_score = -video_noise / (video_remaining.clamp_min(epsilon) * noise_variance)

        audio_progress, audio_derivative = audio_clock(start_time, audio_delay)
        audio_progress = rearrange(audio_progress, "batch -> batch 1 1")
        audio_derivative = rearrange(audio_derivative, "batch -> batch 1 1")
        audio_velocity_time = torch.where(
            audio_derivative > 0.0,
            audio_progress / audio_derivative.clamp_min(epsilon),
            torch.zeros_like(audio_derivative),
        )
        audio_remaining = 1.0 - audio_progress
        audio_noise = audio_state - audio_velocity_time * audio_velocity
        audio_score = -audio_noise / (audio_remaining.clamp_min(epsilon) * noise_variance)

        video_diffusion = diffusion_scale * video_remaining
        audio_diffusion = diffusion_scale * audio_remaining
        video_drift = video_velocity + 0.5 * video_diffusion.square() * video_score
        audio_drift = audio_velocity + 0.5 * audio_diffusion.square() * audio_score
        step_size = end_time - start_time
        video_step = rearrange(step_size, "batch -> batch 1 1 1")
        audio_step = rearrange(step_size, "batch -> batch 1 1")
        video_state = (
            video_state + video_step * video_drift + video_step.sqrt() * video_diffusion * torch.randn_like(video_state)
        )
        audio_state = (
            audio_state + audio_step * audio_drift + audio_step.sqrt() * audio_diffusion * torch.randn_like(audio_state)
        )
    return video_state, audio_state

### Stochastic Interpolant

In [ ]:
class AudioVisualFlowTransport:
    def __init__(self, config: TransportConfig) -> None:
        self.config = config

    @classmethod
    def from_omegaconf(cls, config: DictConfig) -> Self:
        values = OmegaConf.to_container(config, resolve=True)
        return cls(TransportConfig.model_validate(values))

    @jaxtyped(typechecker=beartype)
    def shift_time(self, time: Float[Tensor, "..."]) -> Float[Tensor, "..."]:
        shift = self.config.time_distribution.shift
        return shift * time / (1.0 + (shift - 1.0) * time)

    @jaxtyped(typechecker=beartype)
    def sample_time(self, reference: Float[Tensor, "batch ..."]) -> Float[Tensor, "batch"]:  # noqa: F821
        """
        given reference tensor and distribution (passed by cfg),
        construct a time-series following that reference.
        similar to torch/np.linspace but dist instead of axes
        returns a sigmoid offset by the 1st and 2nd moments of the dist.
        helpful docs are torch.randn/rand for normal and unif, respectively.
        """
        distribution = self.config.time_distribution
        if distribution.type == "uniform":
            time = torch.rand(
                reference.shape[0],
                device=reference.device,
                dtype=reference.dtype,
            )
        else:
            normal = torch.randn(
                reference.shape[0],
                device=reference.device,
                dtype=reference.dtype,
            )
            time = torch.sigmoid(distribution.mean + distribution.standard_deviation * normal)
        return self.shift_time(time)

    @jaxtyped(typechecker=beartype)
    def sample_interpolant(
        self,
        target_video: Float[Tensor, "batch time patch video_feature"],
        target_audio: Float[Tensor, "batch time audio_feature"],
    ) -> AudioVisualInterpolant:
        if target_video.shape[:2] != target_audio.shape[:2]:
            raise ValueError("Transport audio and video targets must align")
        time = self.sample_time(target_video)
        # we need tensor shapes to match, same as np.reshape(reference_tensor[-1])
        video_time = rearrange(time, "batch -> batch 1 1 1")
        audio_time, audio_derivative = audio_clock(time, self.config.audio_delay)
        # see above comment
        audio_time = rearrange(audio_time, "batch -> batch 1 1")
        audio_derivative = rearrange(audio_derivative, "batch -> batch 1 1")
        # subject to change, but usually a constant sigma is ok, we aren't doing VP/VESDE here
        # TODO: investigate if VP/VE SDE with a min and max sigma performs better here
        video_noise = self.config.noise_standard_deviation * torch.randn_like(target_video)
        audio_noise = self.config.noise_standard_deviation * torch.randn_like(target_audio)
        # SI formula: x + (t-s)v_{s,t}(x)
        return AudioVisualInterpolant(
            video_state=(1.0 - video_time) * video_noise + video_time * target_video,
            audio_state=(1.0 - audio_time) * audio_noise + audio_time * target_audio,
            video_velocity=target_video - video_noise,
            audio_velocity=audio_derivative * (target_audio - audio_noise),
            time=time,
        )

    @jaxtyped(typechecker=beartype)
    def sample_end_time(
        self,
        start_time: Float[Tensor, "batch"],  # noqa: F821
        maximum_time_gap: float,
    ) -> Float[Tensor, "batch"]:  # noqa: F821
        if not 0.0 < maximum_time_gap <= 1.0:
            raise ValueError("Maximum flow-map time gap must be in (0, 1]")
        return start_time + torch.rand_like(start_time) * (1.0 - start_time) * maximum_time_gap

    # what we call diag_frac. see fig 2  in https://arxiv.org/abs/2505.18825v2 for a nice visual
    @jaxtyped(typechecker=beartype)
    def time_grid(
        self,
        integration_steps: int,
        reference: Float[Tensor, "batch ..."],
    ) -> Float[Tensor, "grid"]:  # noqa: F821
        if integration_steps <= 0:
            raise ValueError("Flow-map integration steps must be positive")
        uniform_grid = torch.linspace(
            0.0,
            1.0,
            integration_steps + 1,
            device=reference.device,
            dtype=reference.dtype,
        )
        return self.shift_time(uniform_grid)

### Loss Function

In [ ]:
@jaxtyped(typechecker=beartype)
def flow_matching_loss(
    transport: AudioVisualFlowTransport,
    predictor: AudioVisualFlowPredictor,
    video_latents: Float[Tensor, "batch time patch video_feature"],
    audio_latents: Float[Tensor, "batch time audio_feature"],
    context_steps: int,
    actions: Float[Tensor, "batch target_time action"] | None,
    modality_presence: Float[Tensor, "batch 2"] | None = None,
    context_presence: Float[Tensor, "batch 2"] | None = None,
) -> Float[Tensor, ""]:
    """
    Follows the Lipman, et. al formulation of flow matching. However, the comp graph is slightly different

    effectively: model spits out audio and video latents. up to some context steps, context audio and video is retained.

    these are stopgradded for the LSD formulation from Boffi, et. al

    the comp graph, as a result looks like

    Eq 27 from Boffi Et. Al for each modality.


    @param transport: class which has sampling and solver for flow matching
    @param predictor: z_t predictor (DiTDH)which is our actual "world model" end state. this is supposed to do latent rollouts
    @param video_latents: float tensor representing the outputs of passing video through encoder. shape batch x timesteps x patch x video_feature (or feature, like we call it later)
    @param audio_latents float tensor representing results of passing waveform through audio encoder of shape batch x timesteps x audio_features
    @param context_steps: int. meant for a "window", or cutoff for how much ctx to provide. Will need to find happy medium for this and action conditioning.
    @param actions: optional float tensor of shape batch x target_time x action meant to learn latent action for a LAM. might deprecate

    Returns:
        Float tensor with 1 entry, representing loss. video loss + audio loss
    """
    # ctx detach for rhs of flow matching
    context_video = video_latents[:, :context_steps].detach()
    context_audio = audio_latents[:, :context_steps].detach()

    # sg for LSD
    target_video = video_latents[:, context_steps:].detach()
    target_audio = audio_latents[:, context_steps:].detach()

    # (s,t)
    interpolant = transport.sample_interpolant(target_video, target_audio)
    video_state = interpolant.video_state
    audio_state = interpolant.audio_state
    if modality_presence is not None:
        if not torch.all((modality_presence == 0) | (modality_presence == 1)) or not torch.all(
            modality_presence.sum(-1) > 0
        ):
            raise ValueError("Every example must contain at least one declared observed modality")
        video_mask = modality_presence[:, 0, None, None, None]
        audio_mask = modality_presence[:, 1, None, None]
        context_video = context_video * video_mask
        context_audio = context_audio * audio_mask
        video_state = video_state * video_mask
        audio_state = audio_state * audio_mask
    if context_presence is not None:
        context_video = context_video * context_presence[:, 0, None, None, None]
        context_audio = context_audio * context_presence[:, 1, None, None]
    # \dot(v) \forall{m}
    video_velocity, audio_velocity = predictor(
        video_state,
        audio_state,
        interpolant.time,
        interpolant.time,
        context_video,
        context_audio,
        actions,
    )

    # super easy MSE
    if modality_presence is not None:
        video_error = reduce(
            (video_velocity - interpolant.video_velocity).square(), "batch time patch feature -> batch", "mean"
        )
        audio_error = reduce(
            (audio_velocity - interpolant.audio_velocity).square(), "batch time feature -> batch", "mean"
        )
        return (video_error * modality_presence[:, 0]).sum() / modality_presence[:, 0].sum().clamp_min(1) + (
            audio_error * modality_presence[:, 1]
        ).sum() / modality_presence[:, 1].sum().clamp_min(1)
    return functional.mse_loss(video_velocity, interpolant.video_velocity) + functional.mse_loss(
        audio_velocity, interpolant.audio_velocity
    )


@jaxtyped(typechecker=beartype)
def slim_lsd_loss(
    transport: AudioVisualFlowTransport,
    predictor: AudioVisualFlowPredictor,
    video_latents: Float[Tensor, "batch time patch video_feature"],
    audio_latents: Float[Tensor, "batch time audio_feature"],
    context_steps: int,
    actions: Float[Tensor, "batch target_time action"] | None,
    maximum_time_gap: float,
) -> Float[Tensor, ""]:
    """
    See above loss, but uses the alternative sg formulation of https://arxiv.org/abs/2609.16222v1 for 1/2 the amount of NFEs during fwd prop. EMA teacher allows for self-distillation (obviously) but currently needs the MATH backend for SDPA which is NOT ideal.

    @param transport: class which has sampling and solver for flow matching
    @param predictor: z_t predictor (DiTDH)which is our actual "world model" end state. this is supposed to do latent rollouts
    @param video_latents: float tensor representing the outputs of passing video through encoder. shape batch x timesteps x patch x video_feature (or feature, like we call it later)
    @param audio_latents float tensor representing results of passing waveform through audio encoder of shape batch x timesteps x audio_features
    @param context_steps: int. meant for a "window", or cutoff for how much ctx to provide. Will need to find happy medium for this and action conditioning.
    @param maximum_time_gap: how much the space between (u,t) can be in the SI since there is a chance the distribution becomes non-periodic or looses concavity
    """
    context_video = video_latents[:, :context_steps].detach()
    context_audio = audio_latents[:, :context_steps].detach()
    target_video = video_latents[:, context_steps:].detach()
    target_audio = audio_latents[:, context_steps:].detach()
    interpolant = transport.sample_interpolant(target_video, target_audio)
    end_time = transport.sample_end_time(interpolant.time, maximum_time_gap)
    detached_actions = None if actions is None else actions.detach()

    student_video, student_audio = predictor(
        interpolant.video_state,
        interpolant.audio_state,
        interpolant.time,
        end_time,
        context_video,
        context_audio,
        detached_actions,
    )
    # TODO: replace with custom kernel later when have time, i dont think FA2 has jvp
    with torch.no_grad():
        with sdpa_kernel(SDPBackend.MATH):
            predictor_outputs, end_derivatives = torch.func.jvp(
                lambda differentiable_end: predictor(
                    interpolant.video_state,
                    interpolant.audio_state,
                    interpolant.time,
                    differentiable_end,
                    context_video,
                    context_audio,
                    detached_actions,
                ),
                (end_time,),
                (torch.ones_like(end_time),),
            )
        predicted_video, predicted_audio = predictor_outputs
        video_end_derivative, audio_end_derivative = end_derivatives
        if predicted_video.shape != student_video.shape or predicted_audio.shape != student_audio.shape:
            raise RuntimeError("Slim-LSD JVP changed predictor output shapes")
        mapped_video, mapped_audio = flow_map_jump(
            interpolant.video_state,
            interpolant.audio_state,
            student_video.detach(),
            student_audio.detach(),
            interpolant.time,
            end_time,
        )
        teacher_video, teacher_audio = predictor(
            mapped_video,
            mapped_audio,
            end_time,
            end_time,
            context_video,
            context_audio,
            detached_actions,
        )

    tangent_video, tangent_audio = flow_map_jump(
        student_video,
        student_audio,
        video_end_derivative,
        audio_end_derivative,
        interpolant.time,
        end_time,
    )
    video_residual = tangent_video - teacher_video
    audio_residual = tangent_audio - teacher_audio
    return torch.mean(video_residual.square()) + torch.mean(audio_residual.square())

# Pretraining

# Postraining

sample_data
